# Table 3 — HMDA end-to-end (E3)

Paper layout (Table 3): Constr. Classification | Constr. Extraction | Viol. Discovery, each P/R/F1; bold = best per metric and dataset. Only the Norma variants C+MS and Multi-Step (no Document row, no DC Discovery / XiYan-SQL baselines). `fmt` helper copied from `1_run_mapper_eval_full.ipynb`.

In [ ]:
import pandas as pd
import os


In [ ]:
# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

def fmt(val, bold=False):
    try:
        s = f"{float(val):.2f}"
    except (ValueError, TypeError):
        s = "--"
    return f"\\textbf{{{s}}}" if bold else s


def _all_cols(groups):
    return [col for _, cols in groups for col in cols]




In [ ]:
# HMDA results: reads ONLY small result files under results/<HMDA_RESULTS>/ (never data CSVs).
# Environment: HMDA_RESULTS (default "hmda"; e.g. "hmda_v2"), HMDA_MODELS (model manifest).
import os
from pathlib import Path
import pandas as pd
import yaml

ROOT = Path.cwd()
while not (ROOT / "experiments").is_dir():
    ROOT = ROOT.parent
RESULTS_DIR = ROOT / "results" / os.environ.get("HMDA_RESULTS", "hmda")
FIG_DIR = RESULTS_DIR / "figures"
import sys
sys.path.insert(0, str(ROOT / "notebooks" / "results_analysis"))
FIG_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST = yaml.safe_load(
    (ROOT / os.environ.get("HMDA_MODELS", "experiments/configs/hmda_models.yaml")).read_text()
)

HMDA_DATASET_ORDER = ["hmdaz", "hmda_checks"]
HMDA_DATASET_LABELS = {"hmdaz": "HMDAZ", "hmda_checks": "HMDA Checks"}
# Model order/labels follow the manifest (paper_name)
HMDA_MODEL_LABELS = {model["slug"]: model["paper_name"] for model in MANIFEST["models"]}
print(f"Results: {RESULTS_DIR}")


def load_hmda_results(experiment_dir: str) -> pd.DataFrame:
    """Concatenate results_summary.csv of every run; per variant keep the latest
    successful attempt (else the latest failure)."""
    frames = []
    for path in sorted((RESULTS_DIR / experiment_dir).glob("*/results_summary.csv")):
        frame = pd.read_csv(path)
        frame["run_dir"] = path.parent.name
        frames.append(frame)
    if not frames:
        raise FileNotFoundError(f"No results under {RESULTS_DIR / experiment_dir}")
    results = pd.concat(frames, ignore_index=True)
    results["_ok"] = results["status"] == "success"
    results = results.sort_values(["_ok", "run_dir"])
    results = results.drop_duplicates(["dataset", "variant_name"], keep="last").drop(columns="_ok")
    failed = results[results["status"] != "success"]
    if not failed.empty:
        print("Failed variants (excluded from plots):")
        print(failed[["dataset", "variant_name", "status"]].to_string(index=False))
    return results


In [ ]:
# Table 3 (paper layout); module: notebooks/results_analysis/hmda_latex.py
from hmda_latex import TABLE3_GROUPS, table3

METHOD_ORDER = ["C+MS", "Multi-Step"]  # Norma variants only (no Document, no baselines)
METRIC_COLS = [c for _, cols in TABLE3_GROUPS for c in cols]

df = load_hmda_results("e3_end2end")
df["method"] = df["variant_name"].str.split(" | ", regex=False).str[0]
df = df[(df["status"] == "success") & df["method"].isin(METHOD_ORDER)].copy()

latex = table3(df, METHOD_ORDER, HMDA_DATASET_ORDER, HMDA_DATASET_LABELS)
(FIG_DIR / "hmda_table3_end2end.tex").write_text(latex, encoding="utf-8")
df[["dataset", "method", *METRIC_COLS]].to_csv(FIG_DIR / "hmda_table3_end2end.csv", index=False)
display(df[["dataset", "method", *METRIC_COLS]].round(2))
print(latex)